In [ ]:
import pandas as pd
from common import *

In [ ]:
main_dir = "data/home-credit-credit-risk-model-stability/parquet_files/"
main_dir_train = f"{main_dir}train/"

In [ ]:
credit_bureau_names = ["train_credit_bureau_a_2_0", "train_credit_bureau_a_2_1",
                       "train_credit_bureau_a_2_2", "train_credit_bureau_a_2_3",
                       "train_credit_bureau_a_2_4", "train_credit_bureau_a_2_5",
                       "train_credit_bureau_a_2_6", "train_credit_bureau_a_2_7",
                       "train_credit_bureau_a_2_8", "train_credit_bureau_a_2_9",
                       "train_credit_bureau_a_2_10"]
train_credit_bureau = []
for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_train}{credit_bureau}.parquet")
    numerical, categorical, dates = [], [], []
    for col in data.columns.difference(["case_id"]):
        dtype = str(data[col].dtype)
        if "date" in col:
            dates.append(col)
        elif "int" in dtype or "float" in dtype:
            numerical.append(col)
        else:
            categorical.append(col)
    numerical_data = data[["case_id"]+numerical].groupby(["case_id"]).agg(["max", "min", "mean", "count"]).reset_index()
    numerical_data.columns = ["_".join(x) for x in numerical_data.columns]
    data = numerical_data
    train_credit_bureau.append(data)

In [ ]:
data = pd.read_parquet(f"{main_dir_train}train_credit_bureau_a_1_0.parquet")
data_2 = pd.read_parquet(f"{main_dir_train}train_credit_bureau_a_2_0.parquet")
data_3 = pd.read_parquet(f"{main_dir_train}train_credit_bureau_b_1.parquet")

In [ ]:
data_ds = get_column_descriptions(data)
data_2_ds = get_column_descriptions(data_2)
data_3_ds = get_column_descriptions(data_3)

In [ ]:
credit_bureau_names = ["train_credit_bureau_a_2_0", "train_credit_bureau_a_2_1",
                       "train_credit_bureau_a_2_2", "train_credit_bureau_a_2_3",
                       "train_credit_bureau_a_2_4", "train_credit_bureau_a_2_5",
                       "train_credit_bureau_a_2_6", "train_credit_bureau_a_2_7",
                       "train_credit_bureau_a_2_8", "train_credit_bureau_a_2_9",
                       "train_credit_bureau_a_2_10"]

credit_bureau_a_2, credit_bureau_a_2_c = [], []
import gc
for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_train}{credit_bureau}.parquet")

    # Preprocess

    # Categorical dataset
    # data_object = get_dataset_by_datatype(data, "categorical", extra_columns=["case_id"])
    # data_object = data_object.replace({"a55475b1":pd.NA})
    # data_object = categorical_to_dummies(data_object).groupby(["case_id"]).sum()
    # credit_bureau_a_2_c.append(data_object)
    # del data_object
    # gc.collect()

    # Date
    # data_date = get_dataset_by_datatype(data, "date", extra_columns=["case_id"])
    

    # Numerical dataset
    data_numerical = get_dataset_by_datatype(data, "numerical", extra_columns=["case_id"],
                                             exclude_columns=["num_group2"])
    del data
    gc.collect()
    data_numerical["entries_credit_bureau"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
    data_numerical = data_numerical.drop(columns=["num_group1"])
    data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min"])
    data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
    data_numerical = data_numerical.rename(columns={"entries_credit_bureau_max":"entries_credit_bureau"}).drop(columns=["entries_credit_bureau_min"])
    data_numerical = data_numerical.reset_index()


    
    credit_bureau_a_2.append(data_numerical)
    del data_numerical
    gc.collect()

credit_bureau_a_2 = pd.concat(credit_bureau_a_2)
credit_bureau_a_2 = delete_constant_columns(credit_bureau_a_2)
credit_bureau_a_2 = delete_null_columns(credit_bureau_a_2, 0.1)

In [ ]:
credit_bureau_names = ["train_credit_bureau_a_1_0", "train_credit_bureau_a_1_1",
                       "train_credit_bureau_a_1_2", "train_credit_bureau_a_1_3"]

credit_bureau_a_1, credit_bureau_a_2_c = [], []
import gc
for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_train}{credit_bureau}.parquet")

    # Preprocess

    # Categorical dataset
    # data_object = get_dataset_by_datatype(data, "categorical", extra_columns=["case_id"])
    # data_object = data_object.replace({"a55475b1":pd.NA})
    # data_object = categorical_to_dummies(data_object).groupby(["case_id"]).sum()
    # credit_bureau_a_2_c.append(data_object)
    # del data_object
    # gc.collect()

    # Date
    # data_date = get_dataset_by_datatype(data, "date", extra_columns=["case_id"])
    

    # Numerical dataset
    data_numerical = get_dataset_by_datatype(data, "numerical", extra_columns=["case_id"],
                                            exclude_columns=["num_group2"])
    data_numerical["entries_credit_bureau_1"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
    data_numerical = data_numerical.drop(columns=["num_group1"])
    data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min", "mean"])
    data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
    data_numerical = data_numerical.rename(columns={"entries_credit_bureau_1_max":"entries_credit_bureau_1"}).drop(columns=["entries_credit_bureau_1_min", "entries_credit_bureau_1_mean"])

    
    credit_bureau_a_1.append(data_numerical)
    del data_numerical
    gc.collect()

credit_bureau_a_1 = pd.concat(credit_bureau_a_1)
# credit_bureau_a_1 = delete_constant_columns(credit_bureau_a_1)
# credit_bureau_a_1 = delete_null_columns(credit_bureau_a_1, 0.1)

In [ ]:
plot_null_percent(credit_bureau_a_1)

In [ ]:
# delete_constant_columns(credit_bureau_a_1)
delete_null_columns(credit_bureau_a_1, 0.4)

In [ ]:
(credit_bureau_a_1["debtoutstand_525A_max"] == credit_bureau_a_1["debtoutstand_525A_min"]).sum()

In [ ]:
credit_bureau_a1 = pd.read_parquet(f"{main_dir_train}train_credit_bureau_a_1_0.parquet")

In [ ]:
data_object = get_dataset_by_datatype(credit_bureau_a1, "categorical", extra_columns=["case_id"])
data_object = data_object.replace({"a55475b1":pd.NA})
# data_object = categorical_to_dummies(data_object).groupby(["case_id"]).sum()
# credit_bureau_a_2_c.append(data_object)
# del data_object
# gc.collect()

In [ ]:
data_numerical = get_dataset_by_datatype(credit_bureau_a1, "numerical", extra_columns=["case_id"],
                                         exclude_columns=["num_group2"])
data_numerical["entries_credit_bureau_1"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
data_numerical = data_numerical.drop(columns=["num_group1"])
data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min", "mean"])
data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
data_numerical = data_numerical.rename(columns={"entries_credit_bureau_1_max":"entries_credit_bureau_1"}).drop(columns=["entries_credit_bureau_1_min", "entries_credit_bureau_1_mean"])
# data_numerical = data_numerical.reset_index()
data_numerical

In [ ]:
credit_bureau_b1 = pd.read_parquet(f"{main_dir_train}train_credit_bureau_b_1.parquet")
credit_bureau_b1.groupby(["case_id"]).sum().shape[0]

In [ ]:
credit_bureau_b2 = pd.read_parquet(f"{main_dir_train}train_credit_bureau_b_2.parquet")
credit_bureau_b2_desc = get_column_descriptions(credit_bureau_b2)
convert_date_columns(credit_bureau_b2, ["pmts_date_1107D"])
credit_bureau_b2.groupby(["case_id"]).count().shape[0]

In [ ]:
train_credit_bureau = pd.concat(train_credit_bureau)

In [ ]:
plot_null_percent(train_credit_bureau, 0.1)
delete_null_columns(train_credit_bureau, 0.2)

In [ ]:
train_credit_bureau[1].merge(train_credit_bureau[0], on=["case_id"])

In [ ]:
descrs = get_column_descriptions(train_credit_bureau[0])
descrs